### Chapter 7: Search In-Depth
### Topic 4: Hybrid Search — BM25 + Dense via Reciprocal Rank Fusion (RRF)

### Why Do We Need Hybrid Search?

We already have two retrieval methods:

- **BM25:** Good at matching exact keywords.
- **Dense Retrieval:** Good at matching semantic meaning.

Each has strengths and weaknesses.

---

### BM25 Problem

User searches:

```text
close my fixed deposit early
```

Document:

```text
Premature withdrawal of FD is allowed.
```

BM25 looks for exact words.

It cannot understand:

- close = premature withdrawal
- fixed deposit = FD
- early = before maturity

So it gives a low score.

---

### Dense Retrieval Problem

Dense Retrieval understands meaning.

It correctly realizes:

```text
close my fixed deposit early
```

and

```text
Premature withdrawal of FD is allowed.
```

mean almost the same thing.

However, Dense Retrieval may sometimes confuse documents with similar meanings.

For example:

```text
Document 1
Premature withdrawal of FD is allowed.

Document 2
Loan foreclosure is allowed.
```

Both discuss **closing a financial product early**.

Their embeddings may be similar, causing Dense Retrieval to rank both highly, even though only the first document is about FDs.

---

### Idea Behind Hybrid Search

Instead of choosing one method, use both.

BM25 produces one ranked list.

```text
1. Document A
2. Document C
3. Document B
```

Dense Retrieval produces another ranked list.

```text
1. Document C
2. Document A
3. Document D
```

Now we combine these two rankings.

This is called **Hybrid Search**.

---

### Why Can't We Average the Scores?

Suppose BM25 gives:

```text
Document A = 18.5
Document B = 7.3
```

Dense Retrieval gives:

```text
Document A = 0.92
Document B = 0.81
```

These numbers mean completely different things.

- BM25 scores depend on keyword frequency.
- Dense Retrieval scores are cosine similarities between embeddings.

Adding them together would be meaningless because they are measured on different scales.

---

### Solution: Reciprocal Rank Fusion (RRF)

Instead of combining the scores, RRF combines the **ranks**.

Suppose BM25 ranks:

```text
Document A = Rank 1
Document B = Rank 2
Document C = Rank 3
```

Dense Retrieval ranks:

```text
Document C = Rank 1
Document A = Rank 2
Document D = Rank 3
```

RRF ignores the actual scores.

It only asks:

> **"Did this document rank highly in one or both retrieval methods?"**

A document that appears near the top of both lists receives a high final score.

---

### Why RRF Works

BM25 contributes:

- Exact keyword matching.

Dense Retrieval contributes:

- Semantic understanding.

RRF combines their rankings without worrying about incompatible score scales.

This produces better retrieval quality while requiring **no additional model training**.

---

### Key Idea

- **BM25:** Exact word matching.
- **Dense Retrieval:** Semantic matching.
- **Hybrid Search:** Uses both together.
- **RRF:** Merges the ranked lists using document positions instead of raw scores.

---


### Internal Working — Step by Step

The idea behind RRF is simple:

> **If multiple retrieval methods rank a document highly, it is probably a good result.**

Instead of comparing their scores, RRF compares their **positions (ranks)**.

---

### Step 1: BM25 Produces a Ranking

Suppose BM25 returns:

```text
Rank  Document

1     A
2     B
3     C
```

BM25 thinks **A** is the best match.

---

### Step 2: Dense Retrieval Produces Another Ranking

Dense Retrieval returns:

```text
Rank  Document

1     C
2     A
3     D
```

Dense Retrieval thinks **C** is the best match.

---

### Step 3: Give Every Rank a Small Score

RRF converts each rank into a small score.

It uses:

```text
Score = 1 / (k + Rank)
```

Usually:

```text
k = 60
```

So:

```text
Rank 1

1 / (60 + 1)

≈ 0.0164
```

```text
Rank 2

1 / (60 + 2)

≈ 0.0161
```

```text
Rank 3

1 / (60 + 3)

≈ 0.0159
```

Notice that the scores are very close.

Being ranked first is only slightly better than being ranked second.

---

### Step 4: Add the Scores

Now add the scores from both retrieval methods.

Document A

```text
BM25
Rank 1
0.0164

Dense
Rank 2
0.0161

Total

0.0325
```

Document C

```text
BM25
Rank 3
0.0159

Dense
Rank 1
0.0164

Total

0.0323
```

Document B

```text
BM25
Rank 2
0.0161

Dense
Not Present
0

Total

0.0161
```

---

### Step 5: Final Ranking

Now sort by the total score.

```text
Rank

1. Document A
2. Document C
3. Document B
```

Notice something interesting.

Neither BM25 nor Dense Retrieval ranked **A** first.

But **both agreed that A was near the top**.

RRF rewards this agreement.

---

### Why Use k = 60?

Suppose we didn't use 60.

Rank 1 would receive:

```text
1 / 1 = 1.0
```

Rank 2 would receive:

```text
1 / 2 = 0.5
```

Rank 1 becomes **twice as important** as Rank 2.

That gives too much power to whichever retrieval method happened to rank a document first.

Instead we use:

```text
k = 60
```

Now:

```text
Rank 1 = 0.0164

Rank 2 = 0.0161
```

The difference is very small.

Both retrieval methods contribute fairly.

---

### Why Not Average BM25 and Dense Scores?

Suppose:

```text
BM25 Score = 18.5

Dense Score = 0.92
```

These numbers are completely different kinds of measurements.

Adding them together is meaningless.

Instead, RRF ignores the scores and uses only the rankings:

```text
BM25
Rank 1

Dense
Rank 2
```

Ranks are always comparable because every retrieval method orders documents from **most relevant to least relevant**.

This is why RRF combines **ranks**, not **scores**.

--- 

### The Easiest Way to Remember RRF

Imagine two judges in a singing competition.

- **Judge 1** gives scores out of **100**.
- **Judge 2** gives scores out of **10**.

Suppose a contestant receives:

```text
Judge 1 = 95
Judge 2 = 9.2
```

You **cannot average** these scores directly because they are on different scales.

Instead, ask each judge to **rank** the contestants.

**Judge 1**

```text
1st
2nd
3rd
```

**Judge 2**

```text
1st
2nd
3rd
```

Now both judges use the same scale—**positions (ranks)**—so their rankings can be combined fairly.

**RRF follows the same idea.**

- **BM25** produces one ranked list.
- **Dense Retrieval** produces another ranked list.
- **RRF ignores their raw scores** because they are on different scales.
- **RRF combines only the ranks** to produce the final ranking.

---

### Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Documents appearing in only one list is expected, not a bug:** a document ranked 1st by BM25 but absent from dense's results still gets a meaningful contribution from BM25 alone, and vice versa. This asymmetric coverage is the entire reason hybrid search outperforms either retriever alone.
- **Choosing how many results to fuse from each retriever:** if each retriever only returns its top-3 before fusion, some genuinely relevant documents ranked 4th or 5th by either retriever get excluded before fusion even has a chance to consider them. Common practice: retrieve a wider top-10 to top-20 from each retriever, fuse everything, then trim down to the smaller final top-k actually needed downstream.
- **Latency:** since both BM25 search and dense search are each very fast individually, and RRF fusion itself is just simple in-memory merging over a small number of documents, the total added latency from hybrid fusion is negligible — effectively the same as running either retriever alone, especially if the two searches run in parallel.
- **Cost:** zero additional cost beyond what's already needed to run BM25 and dense retrieval separately — no new model, no new API calls, no new infrastructure. This is exactly why it's the correct first step before considering the more expensive architectures from Topic 3.
- **Monitoring:** track how often the final top-1 result came from BM25's rank 1, dense's rank 1, both, or neither — this reveals which retriever is actually carrying more weight for the real query distribution. Also track how often BM25 returned nothing useful but hybrid still surfaced a relevant document via dense — this directly quantifies RRF's value.
- **Debugging a bad hybrid result:** first check both individual ranked lists — is the correct document present in either list at all? If it's absent from both, this is a retrieval coverage problem, not something RRF can fix — no fusion strategy can recover a document that neither retriever found in the first place. If it's present in one list but ranked very low, the fix might be widening the top-k pulled from each retriever before fusion, or reconsidering the k constant.
- **Security:** hybrid search doesn't change any access-control or PII-scoping requirements — any filtering that restricts which records a caller can see must be applied identically on both the BM25 and dense retrieval paths before fusion, or one retriever could leak records that the other correctly filtered out.
- **Deployment:** at small scale, running both retrievers one after another and fusing is simple and fast enough. At larger scale, running both retrievers as parallel calls and fusing once both complete cuts the wall-clock latency roughly in half compared to running them sequentially.


### Design Decisions, Trade-offs, and Real-Time Dilemmas

- **RRF vs directly combining scores with weights:** a weighted combination requires normalizing both score distributions to a comparable range first, which is fragile and needs re-tuning as the corpus grows or changes. RRF requires no normalization and no weight tuning to get started, and stays robust as the corpus changes, at the cost of discarding how *confident* each retriever was — a very strong BM25 match and a barely-passing BM25 match are treated identically if both happen to be ranked 1st.
- **Using the default constant vs tuning it:** the standard default value is a reasonable universal starting point validated across many retrieval benchmarks. A smaller value sharpens preference toward whichever retriever ranked something highest; a larger value flattens rank differences and blends more evenly. The practical guidance: start with the default, and only tune it once a proper evaluation setup (Topic 9) exists to measure whether tuning actually helps.
- **Equal weighting vs favoring one retriever more:** standard RRF treats both retrievers as equally trustworthy. A weighted variant is possible, giving one retriever more influence than the other — for a Hinglish-heavy corpus, dense retrieval might deserve more weight since vocabulary mismatch is the dominant failure mode, but this should be decided from measured evidence on an evaluation set, not from intuition alone.
- **Two retrievers vs adding a third:** RRF generalizes cleanly to more than two ranked lists — the formula simply sums the rank-based contribution across every retriever included. But a third retriever (like SPLADE from Topic 3) only helps if it actually surfaces correct documents that neither BM25 nor dense already found — the marginal benefit must be measured against the added latency and infrastructure cost before deciding to include it.


### Alternatives and When to Use Each

- **RRF (this topic's approach):** zero training and zero tuning required to get started, robust to the score-scale mismatch problem — the correct default choice for combining two or more ranked retrieval lists, and the right first hybrid strategy for this project.
- **Weighted score fusion with normalization:** can outperform RRF if there's enough labeled data to properly tune the weights and normalization ranges, but is more fragile in production, since those ranges can shift as the corpus grows and need periodic recalibration.
- **Cascade / pre-filter approach (BM25 first, then dense only reranks BM25's results):** faster at very large scale, since dense retrieval only has to run over a small candidate set instead of the whole corpus. The serious risk: any document BM25 scored at exactly zero (the vocabulary mismatch case) gets permanently excluded before dense retrieval ever gets a chance to see it — which is the opposite of what a Hinglish-heavy corpus needs. Not recommended here given the measured severity of vocabulary mismatch; the parallel RRF approach is safer.


### Common Mistakes and Production Failures

- Averaging BM25 and dense scores directly without any normalization — this silently lets whichever score happens to have the larger numeric scale dominate the result, regardless of actual relevance.
- Fusing only a very small top-k from each retriever before combining — this can exclude genuinely relevant documents that either retriever ranked just outside that small cutoff.
- Applying access-control or PII filtering to only one of the two retrieval paths — this can leak records that should have been filtered out, since the other unfiltered path still contributes to the final fused ranking.
- Assuming a document missing from the final fused result is an RRF problem, when it's actually a retrieval coverage problem — if a document wasn't found by either retriever in the first place, no fusion strategy can recover it.
- Adding more retrievers to the fusion (like a third or fourth ranked list) without measuring whether each one actually contributes correct documents the others miss — more retrievers means more latency and more infrastructure to maintain, for potentially no real quality gain.


### Lead-Level Interview Questions

**Basic**

- Q: Why can't you just average a BM25 score and a cosine similarity score together?  
  A: They live on completely different numeric scales — BM25 scores are unbounded and corpus-size dependent, while cosine similarity is bounded between -1 and +1. Averaging them directly lets whichever score has the larger scale dominate, regardless of actual relevance. This is why RRF uses rank position instead of raw scores.

- Q: What does the constant `k` in the RRF formula control?  
  A: It controls how sharply rank position affects the final score. A small k creates a large gap between rank 1 and rank 2; a larger k (like the common default of 60) makes that gap much gentler, so a document ranked highly by one retriever but only moderately by another still contributes meaningfully to the final result.

**Intermediate**

- Q: A document is ranked 1st by BM25 but doesn't appear in the dense retrieval results at all. What happens to it in RRF, and is this a problem?  
  A: It still gets a meaningful score contribution from its BM25 rank alone — this is expected, correct behavior, not a bug. This asymmetric coverage (each retriever contributing where it's strong) is exactly why hybrid search outperforms using either retriever by itself.

- Q: Why is RRF often preferred over weighted score combination in production?  
  A: RRF requires no score normalization and no weight tuning to get started, and stays robust as the corpus changes over time. Weighted combination can outperform RRF if properly tuned with enough labeled data, but is more fragile — normalization ranges can shift as the corpus grows, requiring ongoing recalibration.

**Advanced**

- Q: You're considering a cascade approach — run BM25 first, then only rerank BM25's top results with dense retrieval — instead of RRF's parallel approach. What's the risk for a corpus with significant vocabulary mismatch?  
  A: Any document that BM25 scores at exactly zero because of a vocabulary mismatch would never make it into the candidate set that dense retrieval reranks, permanently excluding it — even though dense retrieval might have found it easily on its own. For a corpus where vocabulary mismatch is the dominant failure mode, this defeats the whole purpose of adding dense retrieval. The parallel RRF approach, where both retrievers see the entire corpus independently, avoids this risk.

- Q: How would you decide whether to add a third retriever (like a learned sparse method) to your RRF fusion?  
  A: Measure its marginal contribution directly — specifically, how often it surfaces genuinely relevant documents that neither of the existing retrievers found at all. If that overlap-adjusted benefit is small, the added latency and infrastructure maintenance cost likely isn't justified. This should be decided from measured evaluation data, not general reputation of the new method.

**Scenario-based**

- Q: Your hybrid system's fused top-1 result is wrong for a specific query. Walk through how you'd debug this.  
  A: First check both individual ranked lists before fusion — is the correct document present in either one at all? If it's absent from both, this is a retrieval coverage problem that no fusion strategy can fix. If it's present in one list but ranked very low, consider whether the top-k pulled from each retriever before fusion was too narrow, or whether the RRF constant needs revisiting — but only after confirming the coverage problem isn't the actual root cause.

**Follow-up questions to expect:**

- "What would you monitor in production to know if RRF is actually helping versus just adding complexity?"
- "How does RRF's behavior change as the corpus grows from a handful of documents to millions?"


### Hidden Concepts / Prerequisites Worth Knowing

- **Rank-based fusion vs score-based fusion is a general pattern, not unique to search:** this same idea — that combining rankings is more robust than combining raw scores from different scales — shows up in other contexts too, like combining rankings from multiple independent judges or metrics.
- **RRF generalizes to any number of ranked lists, not just two:** the formula simply sums the rank-based contribution across as many retrievers as are included, which is why it's easy to later add a third retriever (like the SPLADE approach from Topic 3) without redesigning the fusion logic.
- **The trade-off RRF makes — position over magnitude:** RRF explicitly throws away information about *how confident* each retriever was (its raw score), keeping only *where* it ranked a document. This is a deliberate simplification that trades some potential precision for a large gain in robustness and simplicity.

### Quick Revision Summary (for last-minute interview prep)

> Hybrid search runs BM25 and dense retrieval independently on the same query, each producing its own ranked list, then merges them using Reciprocal Rank Fusion (RRF). RRF combines documents based on their rank position in each list — not their raw scores — because BM25 scores and cosine similarity scores live on incompatible numeric scales and cannot be meaningfully averaged. The formula sums 1/(k + rank) across every retriever a document appears in, with k typically set to 60 to soften the gap between adjacent ranks. This approach requires zero training and zero tuning to start, costs nothing beyond running both retrievers separately, and is the correct first move before considering more expensive architectures like DPR, ColBERT, or SPLADE — it lets each retriever contribute wherever it's strong, so BM25's exact-match reliability and dense retrieval's semantic bridging cover each other's specific, measured weaknesses.


### Module 1: Setup — Two Independent Ranked Lists

Build a BM25 ranked list and a dense-retrieval ranked list (offline TF-IDF+SVD stand-in) over the same knowledge base and query, exactly as Topics 1 and 2 did separately.

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Setup -- two independent ranked lists (BM25 and Dense)
#
# This reuses the exact same techniques from Topic 1 (BM25) and
# Topic 2 (TF-IDF+SVD dense stand-in), just applied together here
# so we have two real ranked lists to fuse.
# ------------------------------------------------------------------

import numpy as np
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

KB_TEXTS = [
    "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.",
    "Fixed Deposit premature closure is allowed subject to applicable penalty charges as per RBI guidelines.",
    "Early exit from your deposit account will attract a deduction from your returns.",
    "The Fixed Deposit interest rate for 24 months is 7.1 percent per annum.",
    "Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures.",
    "Fixed Deposit nomination facility is available for all account holders.",
]

# a Hinglish-flavoured query mixing an exact term ("FD", "penalty") with a
# vocabulary-mismatch phrase ("band karna" = "close/exit", not literally in any doc)
QUERY = "FD band karna hai penalty kitni"

def tokenize(text: str) -> list:
    return text.lower().split()

def normalize_vector(v: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(v)
    return v / norm if norm > 0 else v

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else 0.0

# ---- BM25 ranked list ----
tokenized_corpus = [tokenize(t) for t in KB_TEXTS]
bm25 = BM25Okapi(tokenized_corpus)
bm25_scores = bm25.get_scores(tokenize(QUERY))
bm25_ranked = sorted(range(len(KB_TEXTS)), key=lambda i: bm25_scores[i], reverse=True)

# ---- Dense ranked list (TF-IDF + SVD stand-in, same as Topic 2) ----
vectorizer = TfidfVectorizer()
sparse_matrix = vectorizer.fit_transform(KB_TEXTS)
svd = TruncatedSVD(n_components=5, random_state=42)
dense_vectors = svd.fit_transform(sparse_matrix)
dense_vectors_norm = np.array([normalize_vector(v) for v in dense_vectors])

query_dense_vec = normalize_vector(svd.transform(vectorizer.transform([QUERY]))[0])
dense_scores = [cosine_similarity(query_dense_vec, v) for v in dense_vectors_norm]
dense_ranked = sorted(range(len(KB_TEXTS)), key=lambda i: dense_scores[i], reverse=True)

print("=" * 65)
print("TWO INDEPENDENT RANKED LISTS FOR THE SAME QUERY")
print("=" * 65)
print(f"Query: '{QUERY}'\n")

print("BM25 ranking (by exact term overlap):")
for rank, idx in enumerate(bm25_ranked, start=1):
    print(f"  Rank {rank} | Doc {idx} | score={bm25_scores[idx]:.4f} | {KB_TEXTS[idx][:55]}...")

print("\nDense ranking (by semantic similarity):")
for rank, idx in enumerate(dense_ranked, start=1):
    print(f"  Rank {rank} | Doc {idx} | score={dense_scores[idx]:.4f} | {KB_TEXTS[idx][:55]}...")

print("\nNotice the two rankings mostly agree at the top (Doc 0 wins both),")
print("but diverge further down -- e.g. Doc 3 and Doc 2 swap positions")
print("between BM25 and dense. BM25 rewards exact word overlap ('FD',")
print("'penalty'), dense rewards overall semantic closeness. When they")
print("agree, confidence in the top result is higher; when they disagree,")
print("fusion is what decides -- and both cases are handled by the same")
print("mechanism, which is the point of building it properly.")
print("\nModule 1 complete. Run Module 2 next.")


TWO INDEPENDENT RANKED LISTS FOR THE SAME QUERY
Query: 'FD band karna hai penalty kitni'

BM25 ranking (by exact term overlap):
  Rank 1 | Doc 0 | score=1.8239 | Premature withdrawal of FD incurs a 1 percent penalty o...
  Rank 2 | Doc 1 | score=0.5497 | Fixed Deposit premature closure is allowed subject to a...
  Rank 3 | Doc 2 | score=0.0000 | Early exit from your deposit account will attract a ded...
  Rank 4 | Doc 3 | score=0.0000 | The Fixed Deposit interest rate for 24 months is 7.1 pe...
  Rank 5 | Doc 4 | score=0.0000 | Senior citizens receive an additional 0.5 percent inter...
  Rank 6 | Doc 5 | score=0.0000 | Fixed Deposit nomination facility is available for all ...

Dense ranking (by semantic similarity):
  Rank 1 | Doc 0 | score=0.9322 | Premature withdrawal of FD incurs a 1 percent penalty o...
  Rank 2 | Doc 1 | score=0.5241 | Fixed Deposit premature closure is allowed subject to a...
  Rank 3 | Doc 3 | score=0.4232 | The Fixed Deposit interest rate for 24 months is 7.1 

### Module 2: Why You Cannot Just Average the Raw Scores

Proves, with real numbers from Module 1, why directly combining BM25 and cosine scores produces a meaningless result.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Why raw score averaging is broken
# ------------------------------------------------------------------

print("=" * 65)
print("BROKEN APPROACH: averaging raw BM25 + dense scores directly")
print("=" * 65)
print(f"{'Doc':<5} | {'BM25 score':>11} | {'Dense score':>12} | {'Naive average':>14}")
print("-" * 55)

naive_scores = []
for i in range(len(KB_TEXTS)):
    naive_avg = (bm25_scores[i] + dense_scores[i]) / 2
    naive_scores.append(naive_avg)
    print(f"{i:<5} | {bm25_scores[i]:>11.4f} | {dense_scores[i]:>12.4f} | {naive_avg:>14.4f}")

naive_ranked = sorted(range(len(KB_TEXTS)), key=lambda i: naive_scores[i], reverse=True)
print(f"\nNaive-average ranking: {[('Doc ' + str(i)) for i in naive_ranked]}")

print("\nThe problem: BM25 scores here range roughly", f"{min(bm25_scores):.2f} to {max(bm25_scores):.2f},")
print("while dense scores are squeezed into roughly", f"{min(dense_scores):.2f} to {max(dense_scores):.2f}.")
print("Whichever score has the larger numeric spread will dominate the")
print("average almost regardless of the other retriever's opinion.")
print("This is exactly why we need a fusion method that ignores raw")
print("score magnitude entirely and uses only RANK POSITION instead.")
print("\nModule 2 complete. Run Module 3 next.")


BROKEN APPROACH: averaging raw BM25 + dense scores directly
Doc   |  BM25 score |  Dense score |  Naive average
-------------------------------------------------------
0     |      1.8239 |       0.9322 |         1.3780
1     |      0.5497 |       0.5241 |         0.5369
2     |      0.0000 |       0.0423 |         0.0212
3     |      0.0000 |       0.4232 |         0.2116
4     |      0.0000 |       0.1190 |         0.0595
5     |      0.0000 |      -0.3576 |        -0.1788

Naive-average ranking: ['Doc 0', 'Doc 1', 'Doc 3', 'Doc 4', 'Doc 2', 'Doc 5']

The problem: BM25 scores here range roughly 0.00 to 1.82,
while dense scores are squeezed into roughly -0.36 to 0.93.
Whichever score has the larger numeric spread will dominate the
average almost regardless of the other retriever's opinion.
This is exactly why we need a fusion method that ignores raw
score magnitude entirely and uses only RANK POSITION instead.

Module 2 complete. Run Module 3 next.


### Module 3: Reciprocal Rank Fusion (RRF)

The actual fusion function, built from scratch, applied to the two ranked lists from Module 1.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: Reciprocal Rank Fusion (RRF)
#
# RRF_score(d) = sum over each retriever r of 1 / (k + rank_r(d))
# where rank_r(d) is 1-indexed (1st place, 2nd place, ...)
# ------------------------------------------------------------------

def reciprocal_rank_fusion(ranked_lists: list, k: int = 60) -> dict:
    """
    ranked_lists: a list of ranked lists, where each ranked list is a
                  list of document indices in rank order (best first).
    Returns: dict mapping document index -> combined RRF score.
    """
    rrf_scores = {}
    for ranked_list in ranked_lists:
        for position, doc_id in enumerate(ranked_list, start=1):  # 1-indexed rank
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + position)
    return rrf_scores


fused_scores = reciprocal_rank_fusion([bm25_ranked, dense_ranked], k=60)
fused_ranked = sorted(fused_scores.keys(), key=lambda d: fused_scores[d], reverse=True)

print("=" * 65)
print("RECIPROCAL RANK FUSION (k=60)")
print("=" * 65)
print(f"Query: '{QUERY}'\n")

bm25_rank_lookup = {doc: rank for rank, doc in enumerate(bm25_ranked, start=1)}
dense_rank_lookup = {doc: rank for rank, doc in enumerate(dense_ranked, start=1)}

print(f"{'Fused Rank':<11} | {'Doc':<5} | {'BM25 rank':>9} | {'Dense rank':>10} | {'RRF score':>10}")
print("-" * 60)
for fused_rank, doc_id in enumerate(fused_ranked, start=1):
    b_rank = bm25_rank_lookup[doc_id]
    d_rank = dense_rank_lookup[doc_id]
    print(f"{fused_rank:<11} | {doc_id:<5} | {b_rank:>9} | {d_rank:>10} | {fused_scores[doc_id]:>10.5f}")

print(f"\nFinal fused #1: Doc {fused_ranked[0]} -> {KB_TEXTS[fused_ranked[0]][:65]}...")
print("\nThis final ranking used ONLY rank positions, never the raw BM25")
print("or dense scores -- sidestepping the scale-mismatch problem shown")
print("in Module 2 entirely.")
print("\nModule 3 complete. Run Module 4 next.")


RECIPROCAL RANK FUSION (k=60)
Query: 'FD band karna hai penalty kitni'

Fused Rank  | Doc   | BM25 rank | Dense rank |  RRF score
------------------------------------------------------------
1           | 0     |         1 |          1 |    0.03279
2           | 1     |         2 |          2 |    0.03226
3           | 3     |         4 |          3 |    0.03150
4           | 2     |         3 |          5 |    0.03126
5           | 4     |         5 |          4 |    0.03101
6           | 5     |         6 |          6 |    0.03030

Final fused #1: Doc 0 -> Premature withdrawal of FD incurs a 1 percent penalty on the appl...

This final ranking used ONLY rank positions, never the raw BM25
or dense scores -- sidestepping the scale-mismatch problem shown
in Module 2 entirely.

Module 3 complete. Run Module 4 next.


### Module 4: The `k` Constant — Sharp vs Flat Blending

Shows exactly how changing k changes how much the fusion favors whichever retriever ranked a document highest.

In [4]:

# ------------------------------------------------------------------
# MODULE 4: Effect of the k constant
# ------------------------------------------------------------------

print("=" * 65)
print("EFFECT OF k ON RRF CONTRIBUTION PER RANK POSITION")
print("=" * 65)
print(f"{'Rank':<6} | {'k=1 (sharp)':>12} | {'k=10':>8} | {'k=60 (default)':>15} | {'k=200 (flat)':>13}")
print("-" * 65)

for rank in [1, 2, 3, 5, 10]:
    contributions = [1.0 / (k_val + rank) for k_val in [1, 10, 60, 200]]
    print(f"{rank:<6} | {contributions[0]:>12.4f} | {contributions[1]:>8.4f} | "
          f"{contributions[2]:>15.5f} | {contributions[3]:>13.5f}")

print("\nWith k=1: rank 1 contributes 0.5 and rank 2 contributes 0.333 --")
print("a huge relative gap. A document ranked 1st by ONE retriever can")
print("dominate the fused result almost regardless of the other retriever.")
print("\nWith k=200: rank 1 contributes 0.00498 and rank 2 contributes")
print("0.00495 -- almost identical. Every rank position matters nearly")
print("equally, which flattens out each retriever's individual opinion.")
print("\nk=60 (the common default) sits in between -- meaningful preference")
print("for top ranks, without letting one retriever's rank-1 pick")
print("completely override everything else.")

print("\nSame fusion, compared at different k values on our real lists:")
for k_val in [1, 60, 200]:
    scores_at_k = reciprocal_rank_fusion([bm25_ranked, dense_ranked], k=k_val)
    ranked_at_k = sorted(scores_at_k.keys(), key=lambda d: scores_at_k[d], reverse=True)
    print(f"  k={k_val:<4} -> fused order: {ranked_at_k}")

print("\nModule 4 complete. All Topic 4 modules done.")
print()
print("=" * 65)
print("CHAPTER 7 TOPIC 4 -- KEY POINTS TO REMEMBER")
print("=" * 65)
print("""
  RRF combines ranked lists using RANK POSITION, never raw scores --
  this is what makes it robust to BM25 and dense living on
  incompatible numeric scales.

  RRF_score(d) = sum over retrievers of 1 / (k + rank(d))
  k=60 is the standard default -- balances sharp vs flat blending.

  A document appearing in only ONE list still contributes meaningfully
  -- this is the entire point of hybrid search, not a bug.

  Zero training, zero tuning needed to start -- the correct first
  hybrid strategy before considering DPR/ColBERT/SPLADE (Topic 3).
""")


EFFECT OF k ON RRF CONTRIBUTION PER RANK POSITION
Rank   |  k=1 (sharp) |     k=10 |  k=60 (default) |  k=200 (flat)
-----------------------------------------------------------------
1      |       0.5000 |   0.0909 |         0.01639 |       0.00498
2      |       0.3333 |   0.0833 |         0.01613 |       0.00495
3      |       0.2500 |   0.0769 |         0.01587 |       0.00493
5      |       0.1667 |   0.0667 |         0.01538 |       0.00488
10     |       0.0909 |   0.0500 |         0.01429 |       0.00476

With k=1: rank 1 contributes 0.5 and rank 2 contributes 0.333 --
a huge relative gap. A document ranked 1st by ONE retriever can
dominate the fused result almost regardless of the other retriever.

With k=200: rank 1 contributes 0.00498 and rank 2 contributes
0.00495 -- almost identical. Every rank position matters nearly
equally, which flattens out each retriever's individual opinion.

k=60 (the common default) sits in between -- meaningful preference
for top ranks, without l